In [ ]:
import os
import cv2
import random
import albumentations as A
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
from degradations.degradation import create_degradation_pipeline

INPUT_DIR = 'Datasets' 
OUTPUT_DIR = './dataset_patches' 
NUM_PATCHES = 10000
PATCH_SIZE = 256

print(f"Поиск всех .png изображений в папке: {INPUT_DIR}")
all_image_paths = list(Path(INPUT_DIR).rglob('*.png'))

if not all_image_paths:
    print(f"Ошибка: В папке '{INPUT_DIR}' и ее подпапках не найдено ни одного .png файла.")
else:
    print(f"Найдено изображений: {len(all_image_paths)}")

    transform = create_degradation_pipeline()
    CLEAN_DIR = os.path.join(OUTPUT_DIR, 'clean')
    DEGRADED_DIR = os.path.join(OUTPUT_DIR, 'degraded')
    
    os.makedirs(CLEAN_DIR, exist_ok=True)
    os.makedirs(DEGRADED_DIR, exist_ok=True)
    
    print(f"Чистые патчи будут сохранены в: {CLEAN_DIR}")
    print(f"Искаженные патчи будут сохранены в: {DEGRADED_DIR}")

    for i in tqdm(range(NUM_PATCHES), desc="Создание пар патчей"):
        
        patch_created = False
        while not patch_created:
            try:
                random_image_path = random.choice(all_image_paths)
                image = cv2.imread(str(random_image_path))
                
                if image is None:
                    print(f"Предупреждение: не удалось прочитать файл {random_image_path}, пропускаем.")
                    continue
                
                h, w, _ = image.shape
                if h < PATCH_SIZE or w < PATCH_SIZE:
                    continue

                max_x = w - PATCH_SIZE
                max_y = h - PATCH_SIZE
                start_x = random.randint(0, max_x)
                start_y = random.randint(0, max_y)
                
                clean_patch = image[start_y : start_y + PATCH_SIZE, start_x : start_x + PATCH_SIZE]

                augmented = transform(image=clean_patch)
                degraded_patch = augmented['image']

                output_filename = f"{i}.png"
                
                clean_output_path = os.path.join(CLEAN_DIR, output_filename)
                cv2.imwrite(clean_output_path, clean_patch)
                
                degraded_output_path = os.path.join(DEGRADED_DIR, output_filename)
                cv2.imwrite(degraded_output_path, degraded_patch)
                
                patch_created = True

            except Exception as e:
                print(f"Произошла ошибка при обработке файла {random_image_path}: {e}. Пробуем другой файл.")